# Atelier Novalia : du risque de départ à une décision dans le SI
**Formation 4-020 | 100 % en français | Données entièrement synthétiques.**

Objectifs : cadrer une classification, comparer une référence simple et deux modèles, choisir un seuil sur validation, évaluer une fois sur test et exporter un résultat traçable. Ce TP ne prouve aucune efficacité commerciale réelle.

Durées : exploration et entraînement 40 min ; métriques et seuils 35 min ; restitution Power BI 35 min. Les apports théoriques s'intercalent entre ces étapes.

Unité : client actif à une date fictive de référence. Cible : départ dans les 30 jours suivants. Chaque client apparaît une seule fois. La séparation aléatoire illustre des cohortes indépendantes ; une vraie prévision temporelle nécessite une validation chronologique.

**Consigne :** exécuter les cellules dans l'ordre. Les cellules « Votre analyse » sont les réponses attendues. Les sorties montrent des faits calculés sur une simulation, pas des résultats d'entreprise.

## 1. Charger et vérifier
Le dossier du notebook doit contenir `novalia_clients.csv`. Aucun service en ligne ni clé d'API n'est nécessaire après installation des dépendances.

In [1]:
from pathlib import Path
import json, platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, average_precision_score, confusion_matrix)
SEED = 4020
DOSSIER = Path.cwd()
fichier = DOSSIER / "novalia_clients.csv"
if not fichier.exists():
    raise FileNotFoundError("Placez le notebook à côté de novalia_clients.csv.")
df = pd.read_csv(fichier)
assert df.customer_id.is_unique, "Un client apparaît plusieurs fois."
assert set(df.churn.unique()) == {0, 1}, "La cible doit être binaire."
print("Python :", platform.python_version(), "| scikit-learn :", sklearn.__version__)
print("Dimensions :", df.shape, "| proportion de départs :", round(df.churn.mean(),4))
df.head()

Python : 3.13.5 | scikit-learn : 1.8.0
Dimensions : (5000, 12) | proportion de départs : 0.1748


,customer_id,tenure_months,monthly_spend,support_tickets_90d,late_payments_12m,digital_sessions_30d,contract_type,payment_method,satisfaction_score,discount_rate,products_owned,churn
0,NOV-00000,27,72.10,3,1,16,mensuel,prelevement,7.7,0.05,2,0
1,NOV-00001,32,37.64,1,0,18,mensuel,prelevement,9.6,0.10,3,0
2,NOV-00002,59,125.19,1,1,23,bisannuel,virement,5.6,0.05,3,0
3,NOV-00003,46,80.99,2,0,14,mensuel,prelevement,8.7,0.00,2,0
4,NOV-00004,65,58.10,3,2,16,mensuel,carte,5.7,0.00,3,0


In [2]:
bilan = pd.DataFrame({"type":df.dtypes.astype(str),
                      "manquants":df.isna().sum(),
                      "valeurs_distinctes":df.nunique()})
bilan

,type,manquants,valeurs_distinctes
customer_id,object,0,5000
tenure_months,int64,0,72
monthly_spend,float64,100,4048
support_tickets_90d,int64,0,8
late_payments_12m,int64,0,5
digital_sessions_30d,int64,0,31
contract_type,object,0,3
payment_method,object,0,3
satisfaction_score,float64,100,84
discount_rate,float64,0,4


### Votre analyse 1
Quel est le type de problème ? Quelle colonne est la cible ? Pourquoi exclure l'identifiant ? Que vaut une règle qui prédit toujours « reste » ? Pourquoi la présence d'une association ne prouve-t-elle pas une causalité ?

**Corrigé.** Classification binaire supervisée ; cible churn. L'identifiant ne décrit pas un mécanisme transférable. La règle majoritaire peut afficher une exactitude élevée en ignorant tous les départs. Une association peut être expliquée par une cause commune ou une sélection ; elle ne prouve pas l'effet d'une intervention.

## 2. Séparer avant toute transformation
Nous réservons 60 % à l'entraînement, 20 % à la validation et 20 % au test. Le test reste fermé pendant le choix du modèle et du seuil. La stratification conserve approximativement la fréquence de départ.

In [3]:
numeriques = ["tenure_months","monthly_spend","support_tickets_90d",
              "late_payments_12m","digital_sessions_30d","satisfaction_score",
              "discount_rate","products_owned"]
categorielles = ["contract_type","payment_method"]
variables = numeriques + categorielles
X, y = df[variables], df["churn"]
idx_train, idx_reste = train_test_split(df.index, test_size=.4,
                                    stratify=y, random_state=SEED)
idx_val, idx_test = train_test_split(idx_reste, test_size=.5,
                                  stratify=y.loc[idx_reste], random_state=SEED)
assert not(set(idx_train)&set(idx_val) or set(idx_train)&set(idx_test) or set(idx_val)&set(idx_test))
print({"entrainement":len(idx_train),"validation":len(idx_val),"test":len(idx_test)})

{'entrainement': 3000, 'validation': 1000, 'test': 1000}


## 3. Une chaîne reproductible
L'imputation et la normalisation sont ajustées seulement sur l'entraînement. Le même transformateur est ensuite appliqué à la validation et au test. Une pipeline limite les incohérences ; elle ne détecte pas une variable future introduite par erreur.

In [4]:
def construire_pipeline(classifieur):
    numerique = Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("normaliser",StandardScaler())])
    categoriel = Pipeline([
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("encoder",OneHotEncoder(handle_unknown="ignore"))])
    traitement = ColumnTransformer([
        ("num",numerique,numeriques), ("cat",categoriel,categorielles)])
    return Pipeline([("preparation",traitement),("modele",classifieur)])

modeles = {
    "reference_majoritaire":construire_pipeline(DummyClassifier(strategy="most_frequent")),
    "regression_logistique":construire_pipeline(LogisticRegression(max_iter=1000,random_state=SEED)),
    "foret_aleatoire":construire_pipeline(RandomForestClassifier(
        n_estimators=120,min_samples_leaf=12,max_depth=8,random_state=SEED,n_jobs=1))}
for nom, pipeline in modeles.items():
    pipeline.fit(X.loc[idx_train], y.loc[idx_train])
print("Entraînement terminé. Le test n'a pas été consulté.")

Entraînement terminé. Le test n'a pas été consulté.


## 4. Comparer sur validation
L'Average Precision (AP) résume la courbe précision-rappel. Elle ne représente ni un pourcentage de clients correctement traités ni la valeur économique. Tous les modèles sont comparés sur le même jeu.

In [5]:
def mesurer(y_vrai, scores, seuil=.5):
    predictions=(np.asarray(scores)>=seuil).astype(int)
    return {"exactitude":accuracy_score(y_vrai,predictions),
        "precision":precision_score(y_vrai,predictions,zero_division=0),
        "rappel":recall_score(y_vrai,predictions,zero_division=0),
        "f1":f1_score(y_vrai,predictions,zero_division=0),
        "average_precision":average_precision_score(y_vrai,scores),
        "alertes":int(predictions.sum())}

lignes=[]
for nom,pipeline in modeles.items():
    p=pipeline.predict_proba(X.loc[idx_val])[:,1]
    lignes.append({"modele":nom,**mesurer(y.loc[idx_val],p)})
comparaison=pd.DataFrame(lignes).set_index("modele")
comparaison.round(3)

,exactitude,precision,rappel,f1,average_precision,alertes
modele,,,,,,
reference_majoritaire,0.825,0.000,0.000,0.000,0.175,0
regression_logistique,0.832,0.559,0.189,0.282,0.486,59
foret_aleatoire,0.827,0.571,0.046,0.085,0.460,14


### Votre analyse 2
Quel modèle choisiriez-vous ? Quel autre critère que l'AP peut justifier un modèle plus simple ? Pourquoi ne pas consulter le test pour améliorer ce choix ?

**Corrigé.** Retenir le modèle selon le protocole de validation annoncé, puis discuter lisibilité, latence et exploitation. Le test est une estimation indépendante : s'en servir pour choisir introduit un biais de sélection. Les chiffres du notebook font foi pour cette graine et ces versions.

## 5. Définir le seuil avec la capacité de l'équipe
Hypothèse pédagogique : au plus 10 % des clients peuvent être contactés. Sur les 1 000 clients de validation, cela représente 100 contacts. Le seuil est choisi sur validation, puis gelé.

Attention : une contrainte de quota sur une population future n'est pas garantie par un seuil fixe. Une sélection des k meilleurs scores est une politique différente ; elle doit aussi être documentée.

In [6]:
choisi = comparaison["average_precision"].idxmax()
modele = modeles[choisi]
scores_val = modele.predict_proba(X.loc[idx_val])[:,1]
capacite = int(.10*len(idx_val))
ordonnes=np.sort(scores_val)[::-1]
seuil_capacite=float((ordonnes[capacite-1]+ordonnes[capacite])/2)
seuils=sorted(set([.3,.5,.7,seuil_capacite]))
table_seuils=pd.DataFrame([{"seuil":s,**mesurer(y.loc[idx_val],scores_val,s)} for s in seuils])
print("Modèle retenu sur validation :",choisi)
print("Seuil de capacité choisi sur validation :",round(seuil_capacite,4))
table_seuils.round(3)

Modèle retenu sur validation : regression_logistique
Seuil de capacité choisi sur validation : 0.3836


,seuil,exactitude,precision,rappel,f1,average_precision,alertes
0,0.300,0.833,0.522,0.531,0.527,0.486,178
1,0.384,0.845,0.600,0.343,0.436,0.486,100
2,0.500,0.832,0.559,0.189,0.282,0.486,59
3,0.700,0.828,0.667,0.034,0.065,0.486,9


### Votre analyse 3
Décrivez ce que devient le rappel lorsque le seuil augmente. Qui doit approuver le nombre de contacts et le coût acceptable des erreurs ? Un client à risque élevé est-il nécessairement celui dont le départ peut être évité par une offre ?

**Corrigé.** Un seuil plus élevé réduit les alertes et ne peut augmenter le rappel sur une population fixe. La précision n'est pas nécessairement monotone dans un petit échantillon. Le métier approuve la capacité et le compromis. Risque de départ et effet causal d'une remise ne sont pas équivalents : un essai contrôlé est nécessaire.

## 6. Ouvrir le test une seule fois
Les choix sont maintenant gelés. Si le test est décevant, on ne règle pas le seuil sur ce même test ; on relance une démarche avec un nouveau protocole et, si nécessaire, un nouveau jeu de test.

In [7]:
scores_test=modele.predict_proba(X.loc[idx_test])[:,1]
predictions_test=(scores_test>=seuil_capacite).astype(int)
resultats_test=mesurer(y.loc[idx_test],scores_test,seuil_capacite)
matrice=confusion_matrix(y.loc[idx_test],predictions_test,labels=[0,1])
print("Résultats de test :",{k:round(v,4) if isinstance(v,float) else v for k,v in resultats_test.items()})
pd.DataFrame(matrice,index=["Réel reste","Réel part"],columns=["Prédit reste","Prédit part"])

Résultats de test : {'exactitude': 0.814, 'precision': 0.4476, 'rappel': 0.2686, 'f1': 0.3357, 'average_precision': 0.4792, 'alertes': 105}


,Prédit reste,Prédit part
Réel reste,767,58
Réel part,128,47


## 7. Exporter une décision traçable
Chaque ligne exportée appartient au test indépendant. Le fichier n'est pas une base opérationnelle de prospection. Le score est une estimation issue de la simulation, pas une probabilité garantie ni calibrée pour une vraie entreprise.

In [8]:
export=df.loc[idx_test,["customer_id","contract_type","monthly_spend","churn"]].copy()
export["score_depart"]=scores_test
export["alerte"]=(scores_test>=seuil_capacite).astype(int)
export["seuil"]=seuil_capacite
export["version_modele"]="novalia-demo-1"
export["date_score"]="2026-02-01"
export["cohorte"]="test_independant_synthetique"
export["revenu_expose_estime"]=(export.monthly_spend.fillna(0)*export.score_depart).round(2)
export.to_csv(DOSSIER/"predictions_powerbi.csv",index=False)
comparaison.to_csv(DOSSIER/"comparaison_validation.csv")
table_seuils.to_csv(DOSSIER/"seuils_validation.csv",index=False)
rapport={"n":len(df),"taux_depart":float(df.churn.mean()),
         "modele":choisi,"seuil":seuil_capacite,"validation":comparaison.reset_index().to_dict("records"),
         "test":resultats_test,"matrice":matrice.tolist(),
         "versions":{"numpy":np.__version__,"pandas":pd.__version__,"sklearn":sklearn.__version__}}
(DOSSIER/"resultats_reference.json").write_text(json.dumps(rapport,ensure_ascii=False,indent=2),encoding="utf-8")
print("Exports disponibles pour l'atelier Power BI.")
export.head()

Exports disponibles pour l'atelier Power BI.


,customer_id,contract_type,monthly_spend,churn,score_depart,alerte,seuil,version_modele,date_score,cohorte,revenu_expose_estime
4641,NOV-04641,mensuel,80.59,0,0.038541,0,0.38358,novalia-demo-1,2026-02-01,test_independant_synthetique,3.11
2935,NOV-02935,mensuel,127.51,0,0.596821,1,0.38358,novalia-demo-1,2026-02-01,test_independant_synthetique,76.10
1124,NOV-01124,annuel,51.11,0,0.066030,0,0.38358,novalia-demo-1,2026-02-01,test_independant_synthetique,3.37
4845,NOV-04845,annuel,20.02,0,0.093940,0,0.38358,novalia-demo-1,2026-02-01,test_independant_synthetique,1.88
4942,NOV-04942,mensuel,77.77,1,0.741261,1,0.38358,novalia-demo-1,2026-02-01,test_independant_synthetique,57.65


## 8. Restituer dans Power BI ou dans le support de secours
Importer `predictions_powerbi.csv` avec l'encodage UTF-8 et le séparateur virgule. Définir `score_depart` comme nombre décimal et `alerte` comme entier. Utiliser la locale adaptée aux points décimaux du CSV.

Créer quatre indicateurs : nombre de clients testés ; nombre d'alertes ; montant mensuel observé ; somme pondérée des montants par le score. Le dernier n'est ni du chiffre d'affaires réellement perdu ni une prévision comptable validée.

Ajouter un histogramme des scores, un taux par contrat et une table de segments agrégés. Toujours afficher « cohorte de test synthétique », la version et la date des scores.

Variante sans Power BI : lire le fichier CSV et remplir la fiche A09 dans le cahier. Les objectifs de restitution et de décision restent identiques.

### Votre analyse 4
Quelle action déclenche l'alerte ? Qui est responsable ? Que faire si le modèle n'est pas disponible ? Quel test permettrait de mesurer l'effet réel de l'action commerciale ?

**Corrigé.** Un dossier est envoyé à une file métier avec horodatage, version et politique de sélection. Prévoir un responsable, des règles de non-sollicitation et un mode manuel de secours. Un groupe traité et un groupe témoin comparables, idéalement randomisés, permettent d'estimer l'effet incrémental de la campagne.

## Références techniques
scikit-learn, guide utilisateur : https://scikit-learn.org/stable/user_guide.html

scikit-learn, prévention des fuites : https://scikit-learn.org/1.8/common_pitfalls.html

scikit-learn, métriques : https://scikit-learn.org/stable/modules/model_evaluation.html

Jupyter : https://docs.jupyter.org/en/latest/

Power BI et Python : https://learn.microsoft.com/fr-fr/power-bi/connect-data/desktop-python-scripts

**Limites de la simulation :** loi de génération construite, absence de causalité établie, clients indépendants, pas d'évaluation temporelle réelle, pas de données protégées réelles, pas de connexion à un SI de production.